In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from ugdatalab.models.apogee import Spectrum
from ugdatalab.models.apogee.constants import LABEL_NAMES
from ugdatalab.methods.cannon import CannonModel
from ugdatalab.methods.cannon_likelihood import CannonLabelLikelihood
from ugdatalab.methods.bayesian.mcmc import nuts_sample

In [ ]:
# Load all intermediate products
spec_data = np.load("training_spectra.npz", allow_pickle=True)
model_data = np.load("cannon_model.npz", allow_pickle=True)
cv_data = np.load("cv_results.npz", allow_pickle=True)
nn_data = np.load("nn_results.npz", allow_pickle=True)

# Numerical Results — Single Source of Truth

This notebook re-derives every numerical claim that appears in the report directly from the cached intermediate products (`training_spectra.npz`, `cannon_model.npz`, `cv_results.npz`, `nn_results.npz`). When a number appears here and also in the report, **the value here is authoritative**: the report should match the notebook, not the other way around.

## Training Set Size and Cuts

Re-derived from `training_spectra.npz`. After the four quality cuts described in NB 01 (all five labels finite and non-NULL, SNR > 50, $\log g \leq 4$, $T_{\rm eff} \leq 5700$ K, $[\mathrm{Fe/H}] \geq -1$), the training set contains the number of stars reported below — split 50/50 with seed 40 into a *training* subset that fits the Cannon's polynomial coefficients and a *cross-validation* subset that measures recovery on held-out data. The wavelength range and pixel count are intrinsic properties of the APOGEE log-linear grid (15100–17000 Å on 8575 pixels across three H2RG chips); we do not modify either.

In [ ]:
N_total = spec_data["flux"].shape[0]
n_pixels = spec_data["wavelength"].shape[0]
n_train = len(model_data["train_idx"])
n_cv = len(model_data["cv_idx"])

training_summary = pd.DataFrame({
    "Quantity": [
        "Total training set (after quality cuts)",
        "Training subset (50%)",
        "Cross-validation subset (50%)",
        "Pixels per spectrum",
        "Wavelength range",
    ],
    "Value": [
        f"{N_total}",
        f"{n_train}",
        f"{n_cv}",
        f"{n_pixels}",
        f"{spec_data['wavelength'][0]:.1f}–{spec_data['wavelength'][-1]:.1f} Å",
    ],
    "Source": [
        "NB 01 → training_spectra.npz",
        "NB 02 → cannon_model.npz (train_idx)",
        "NB 02 → cannon_model.npz (cv_idx)",
        "NB 01 → training_spectra.npz",
        "NB 01 → training_spectra.npz",
    ],
})
training_summary


## Pixel Bitmask Statistics

Pixels flagged by the APOGEE bitmask have $\sigma = \infty$ after normalization. Re-derived from `training_spectra.npz`.

The statistics below give the average masked-pixel count across the training set and a worked example for a single star. The mean masked fraction is $\sim 9\%$, dominated by the four inter-chip gaps and the OH airglow region near 15852 Å (see NB 01 Problem 8 for the wavelength attribution). This is geometric, not per-star: the same wavelengths are masked in nearly every star, with cosmic-ray hits and persistent detector glitches contributing the per-star variance. A 9% masking rate would be alarming if pixels were *removed* from the analysis, but in our pipeline they are *down-weighted* via $\sigma_\lambda = 10^6$ — the spectrum stays full-length and the WLS gives them effectively zero contribution. The trained Cannon's per-pixel valid-fraction threshold (NB 02 §6b) handles the chip-edge regions explicitly so the polynomial coefficients are not driven by the few stars whose data extends past the nominal chip boundary.

In [ ]:
flux = spec_data["flux"]
error = spec_data["error"]
apogee_ids = spec_data["apogee_ids"]
n_pixels = flux.shape[1]

# Per-star masked pixel counts (inf error or nan flux)
masked_per_star = np.sum(~np.isfinite(flux) | ~np.isfinite(error), axis=1)
frac_per_star = masked_per_star / n_pixels * 100

# Example star
example_id = "2M21235315+1244123"
ex_idx = int(np.where(apogee_ids == example_id)[0][0])

example_bitmask = pd.DataFrame({
    "Quantity": ["Total pixels", "Masked pixels", "Good pixels", "Masked fraction"],
    "Value": [
        f"{n_pixels}",
        f"{masked_per_star[ex_idx]}",
        f"{n_pixels - masked_per_star[ex_idx]}",
        f"{frac_per_star[ex_idx]:.1f}%",
    ],
    "Source": ["NB 01 → training_spectra.npz"] * 4,
})
print(f"Example star: {example_id}")
display(example_bitmask)

# Across full training set
dataset_bitmask = pd.DataFrame({
    "Statistic": ["Mean masked pixels", "Median masked pixels", "Min masked pixels", "Max masked pixels", "Mean masked fraction"],
    "Value": [
        f"{np.mean(masked_per_star):.1f}",
        f"{np.median(masked_per_star):.0f}",
        f"{np.min(masked_per_star)}",
        f"{np.max(masked_per_star)}",
        f"{np.mean(frac_per_star):.1f}%",
    ],
    "Source": ["NB 01 → training_spectra.npz"] * 5,
})
print(f"\nAcross {len(apogee_ids)} training stars:")
dataset_bitmask


## Surface Gravity Calculation

Re-derived from fundamental constants. Surface gravity is $g = GM/R^2$, so for a $1\,M_\odot$ star

$$\log g = \log g_\odot + \log(M/M_\odot) - 2\log(R/R_\odot) = 4.44 - 2\log(R/R_\odot)\quad\text{(CGS)}.$$

The three rows below are reference values for the main sequence ($R \approx 1\,R_\odot$), the tip of the RGB just before the helium flash ($R \approx 100\,R_\odot$), and the core-helium-burning red clump ($R \approx 15\,R_\odot$).

**Connection to the mystery star.** The mystery-star MCMC fit below returns $\log g \approx 2.01$, which agrees with the analytical red-clump value of 2.09 to within 0.08 dex — well inside the 0.10 dex CV scatter for $\log g$. This is the quantitative basis for identifying the mystery star as a *core-helium-burning red-clump giant* rather than a star ascending the RGB or descending toward the AGB. The same comparison rules out the other reference stages: the mystery star's $\log g$ is far from the main-sequence value of 4.44 (so it is not a dwarf) and far from the RGB-tip value of 0.44 (so it is not at the upper end of the RGB).

In [ ]:
logg_sun = 4.44  # log g of the Sun in CGS

logg_table = pd.DataFrame({
    "Stage": ["Main sequence", "Pre-He flash (tip RGB)", "Core He burning (red clump)"],
    "R / R_sun": [1, 100, 15],
    "log g": [logg_sun - 2 * np.log10(r) for r in [1, 100, 15]],
    "Source": ["analytical (g = GM/R^2)"] * 3,
})
logg_table


## Cannon Model Parameters

Re-derived from `cannon_model.npz`.

The Cannon's parameter budget is `(21 polynomial coefficients + 1 scatter term) × 8575 pixels = 188,650` free parameters in total. This number is large in absolute terms but small *per pixel*: the per-pixel optimization sees only its own column of the flux matrix and the shared design matrix, so each pixel is fit by a 21-parameter polynomial against $\sim 940$ training stars — a comfortably overdetermined problem.

**Training $\chi^2_r \approx 0.99$.** The reduced chi-squared on the training set is essentially unity, meaning the residuals after the trained polynomial-plus-scatter fit are statistically consistent with the combined noise model $\sigma_{n\lambda}^2 + s_\lambda^2$. This is the upper-bound-optimistic measure of model quality (residuals on the same stars used to fit $\boldsymbol{\theta}$); the honest test is the cross-validation table below. We will see in NB 04 that the *MCMC* fit of the mystery star reports $\chi^2_r \approx 10.4$ — a 10× excess that is the quantitative origin of the formal-vs-CV uncertainty discrepancy and is interpreted there as a combination of mystery-star continuum systematics and label-space edge effects.

In [ ]:
n_terms = model_data["theta"].shape[1]
n_pix = model_data["theta"].shape[0]
n_params_total = (n_terms + 1) * n_pix

cannon_summary = pd.DataFrame({
    "Quantity": [
        "Coefficients per pixel",
        "Number of pixels",
        "Total free parameters",
        "Training chi2_r",
    ],
    "Value": [
        f"{n_terms}",
        f"{n_pix}",
        f"{n_params_total:,}",
        f"{float(model_data['chi2_r']):.3f}",
    ],
    "Source": ["NB 02 → cannon_model.npz"] * 4,
})
cannon_summary


## Cross-Validation Bias and Scatter

Re-derived from `cv_results.npz`. **These are the canonical CV recovery numbers** from the original Cannon model (NB 02). Any value cited in the report for "Cannon CV scatter" should match this table exactly.

Reading the table:

- **$T_{\rm eff}$** has the largest absolute scatter (~41 K) but the smallest *fractional* scatter (~1% of the $\sim 4000$ K dynamic range). The Cannon recovers temperature most precisely because $T_{\rm eff}$ has the broadest spectral signature — every line responds to the Boltzmann/Saha balance, and the gradient spectra in NB 02 §8a show that essentially every pixel contributes information about $T_{\rm eff}$.
- **$\log g$** scatter is ~0.10 dex — about 3× larger fractionally than $T_{\rm eff}$. $\log g$ has fewer high-leverage diagnostic features (mainly pressure-broadened line wings and a handful of strong neutral metal cores), and the training set's $\log g$ dynamic range is narrower than the temperature range, so the polynomial coefficients are less tightly constrained.
- **$[\mathrm{Fe/H}]$, $[\mathrm{Mg/Fe}]$, $[\mathrm{Si/Fe}]$** all recover at $\sim 0.033$–$0.034$ dex scatter. These are very tightly localized labels — the gradient spectra show that abundance information lives at a small set of metal-line wavelengths — and the Cannon recovers them with the precision the lab manual targets ($\sim 0.02$–$0.05$ dex).

**Bias is essentially zero for all labels except $T_{\rm eff}$**, which has a small mean offset of $\sim 3$ K — well below the per-star scatter and consistent with sampling noise on the 943-star CV set.

In [ ]:
cannon_fitted = cv_data["fitted_labels"]
true_labels = cv_data["true_labels"]
cannon_resid = cannon_fitted - true_labels

cv_bias_scatter = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Bias (mean offset)": np.mean(cannon_resid, axis=0),
    "Scatter (std)": np.std(cannon_resid, axis=0),
    "Source": ["NB 03 → cv_results.npz"] * len(LABEL_NAMES),
})
cv_bias_scatter


## MCMC Posterior for Mystery Star

Re-derived from scratch by re-running the NUTS sampler with the same seed (42) used in NB 04. The fit takes ~20 s.

The posterior medians and 68% credible intervals are reproduced in the table below. **Physical interpretation** (see NB 04 for the full discussion): the mystery star is a metal-poor ($[\mathrm{Fe/H}] \approx -0.61$), $\alpha$-enhanced ($[\mathrm{Mg/Fe}] \approx +0.30$, $[\mathrm{Si/Fe}] \approx +0.19$) red giant with $T_{\rm eff} \approx 4565$ K and $\log g \approx 2.01$ — within 0.08 dex of the analytical red-clump value derived above. This is the chemical and evolutionary signature of an old, thick-disk or inner-halo, core-helium-burning red-clump star.

**Formal vs. realistic uncertainty.** The 68% CI half-widths in the rightmost column ($\sim 3$ K in $T_{\rm eff}$, $\sim 0.01$ dex in $\log g$, $\sim 0.004$ dex in the abundances) are an order of magnitude smaller than the cross-validation scatter on the table above (~41 K, ~0.10 dex, ~0.033 dex respectively). The CV scatter is the realistic error budget — it captures continuum systematics, line blending, and model inadequacy that the formal MCMC uncertainty cannot. Always quote CV-scatter-sized uncertainties when reporting Cannon labels for a single star, **not** the formal MCMC errors.

In [ ]:
# The MCMC result is produced in NB 04 (terminal analysis).
# To re-derive here, we re-run the MCMC fit with the same seed for reproducibility.

# Reconstruct Cannon model
model = CannonModel(
    theta=model_data["theta"],
    scatter=model_data["scatter"],
    label_names=list(model_data["label_names"]),
    label_means=model_data["label_means"],
    label_stds=model_data["label_stds"],
    wavelength=model_data["wavelength"],
    chi2_r=float(model_data["chi2_r"]),
)

# Read and normalize mystery spectrum
mystery_path = Path("../../course_materials_sp2026/labs/lab_2/mystery_spec_wiped.fits")
continuum_path = Path("continuum_wavelengths.npz")

mystery = Spectrum(mystery_path, continuum_path)
flux_norm = mystery.flux[0]
error_norm = mystery.error[0]

# MCMC fit
lk = CannonLabelLikelihood(x=model.wavelength, y=flux_norm, y_err=error_norm, model=model)
result = nuts_sample(lk, n_steps=2000, n_burn=1000, seed=42)

medians = np.median(result.samples, axis=0)
lo = np.percentile(result.samples, 16, axis=0)
hi = np.percentile(result.samples, 84, axis=0)

mcmc_summary = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Median": medians,
    "16th pct": lo,
    "84th pct": hi,
    "σ (68% CI)": (hi - lo) / 2,
    "Source": ["NB 04 → re-run NUTS (seed=42) on mystery_spec_wiped.fits"] * len(LABEL_NAMES),
})
mcmc_summary


## Neural Network vs Cannon Comparison

Re-derived from `cv_results.npz` and `nn_results.npz`. The table below puts the NN's CV bias and scatter side-by-side with the Cannon's, computed on the *same* held-out CV set with the *same* train/CV split.

**Headline result.** The Cannon outperforms the MLP baseline on every label, with NN scatter $\sim 1.4$–$2.3\times$ larger than the Cannon. The NN also picks up a noticeable bias in $T_{\rm eff}$ (~$-22$ K) and $\log g$ (~$-0.05$ dex) where the Cannon has essentially no bias. Specifically:

- $T_{\rm eff}$: NN scatter $\sim 71$ K vs Cannon $\sim 41$ K (1.75×)
- $\log g$: NN scatter $\sim 0.23$ dex vs Cannon $\sim 0.10$ dex (2.30×)
- $[\mathrm{Fe/H}]$: NN $\sim 0.070$ vs Cannon $\sim 0.033$ (2.10×)
- $[\mathrm{Mg/Fe}]$: NN $\sim 0.061$ vs Cannon $\sim 0.033$ (1.86×)
- $[\mathrm{Si/Fe}]$: NN $\sim 0.047$ vs Cannon $\sim 0.034$ (1.39×)

Root cause analysis is in NB 05. In one sentence: the MLP has no built-in bad-pixel handling, no spectrum-level noise model, a $\sim 5000$:1 parameter:datum ratio, and no physical inductive bias — and these four factors together mean a vanilla MLP at this dataset scale cannot match the Cannon's polynomial-plus-WLS structure, which is well-matched to APOGEE-like spectra.

In [ ]:
nn_fitted = nn_data["nn_fitted_labels"]
nn_true = nn_data["true_labels"]
nn_resid = nn_fitted - nn_true

comparison = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Cannon bias": np.mean(cannon_resid, axis=0),
    "Cannon scatter": np.std(cannon_resid, axis=0),
    "NN bias": np.mean(nn_resid, axis=0),
    "NN scatter": np.std(nn_resid, axis=0),
    "Source": ["NB 03 cv_results.npz + NB 05 nn_results.npz"] * len(LABEL_NAMES),
})
comparison


## Unified Uncertainty Comparison: MCMC vs CV vs NN

The single most informative table in the lab. For each label we put three uncertainties side-by-side:

1. **MCMC formal $\sigma$** — the 68% credible-interval half-width on the mystery star, computed from the NUTS samples above. This is what the per-pixel Gaussian noise model says the label uncertainty should be.
2. **Cannon CV scatter** — the held-out cross-validation residual standard deviation from `cv_results.npz`. This measures how well the Cannon recovers ASPCAP labels on stars it has not been fit to, end-to-end.
3. **NN CV scatter** — the same metric for the MLP from NB 05.

The expected ordering is **MCMC formal < Cannon CV < NN CV**, and the gap between (1) and (2) is the headline systematic error of the Cannon's per-pixel-independence assumption: the formal MCMC error is ~10× too small for $T_{\rm eff}$ and ~10× too small for the abundances. Always quote (2) — the CV scatter — as the realistic error budget for a single star, never (1).

In [ ]:
mcmc_sigma = (hi - lo) / 2  # MCMC 68% CI half-width
cannon_scatter = np.std(cannon_resid, axis=0)
nn_scatter = np.std(nn_resid, axis=0)

unified = pd.DataFrame({
    "Label": LABEL_NAMES,
    "MCMC formal σ": mcmc_sigma,
    "Cannon CV scatter": cannon_scatter,
    "NN CV scatter": nn_scatter,
    "CV / formal": cannon_scatter / mcmc_sigma,
    "NN / Cannon": nn_scatter / cannon_scatter,
    "Source": ["NB 04 NUTS + NB 03 cv_results.npz + NB 05 nn_results.npz"] * len(LABEL_NAMES),
})
unified
